# Notebook 07: Full evaluation metrics for every saved model (no retraining)

Advisor feedback (12 Nov 2026): each run should report the standard metrics of its task.
Every run of Notebooks 05 and 06 saved its **best weights** in `checkpoints/`, so all metrics can be computed again
from the test set **without training anything**. This notebook loads each checkpoint once, runs the 3,000 test images,
keeps the **class probabilities** (softmax) and computes:

| Group | Metric | Why | Reference |
|---|---|---|---|
| Overall | Accuracy | share of correct predictions (misleading alone: 50 % of the test set is No-Anomaly) | Sokolova & Lapalme 2009 |
| | **Balanced accuracy** (= macro recall) | accuracy that weights every class equally | Brodersen et al. 2010 |
| | **Macro precision, macro recall, macro-F1** | every class counts the same → right for imbalanced data | Sokolova & Lapalme 2009; Grandini et al. 2020 |
| | Weighted-F1 | F1 weighted by class size (for comparison with papers that report it) | Grandini et al. 2020 |
| | **MCC** (Matthews correlation coefficient) | one number using the whole confusion matrix; robust to imbalance | Matthews 1975; Chicco & Jurman 2020 |
| | Cohen's κ | agreement beyond chance | Cohen 1960 |
| Threshold-free (from probabilities) | **mAP** = macro average precision (area under each class's precision–recall curve, one-vs-rest) | ranks the scores, not only the top-1 class; PR curves suit imbalanced data | Saito & Rehmsmeier 2015; Everingham et al. 2010 (AP definition) |
| | Macro ROC-AUC (one-vs-rest) | common in the literature, less informative than PR on imbalanced data | Saito & Rehmsmeier 2015 |
| | Top-2 accuracy | correct class among the two highest scores | – |
| Probability quality | **Test loss (NLL)** and **ECE** (expected calibration error, 15 bins) | high validation loss with good accuracy = over-confident model; ECE measures this directly | Guo et al. 2017 |
| Per class | precision, recall, F1, AP, number of test images | where the model fails | – |
| Task-specific | fault vs no-fault: accuracy, precision, **recall** (missed faults), F1 | for maintenance, a missed fault costs more than a false alarm | – |
| | macro-F1 after merging into the 5 groups | the level used for power loss | – |
| Cost | parameters, inference time per image | for running on many drone images | – |
| Comparison with papers | 12-class accuracy, accuracy on the 11 anomaly classes, fault / no-fault accuracy | scored the way each paper reports | Le et al. 2021; Fonseca Alves et al. 2021; Korkmaz & Açıkgöz 2022 |
| Cheap improvements | **test-time augmentation** (average over flips) and **seed ensemble** (average of the 3 seeds) | no retraining; ensembles were used by Le et al. 2021 | – |
| Statistics | **bootstrap 95 % CI** of macro-F1; **McNemar test** vs the best setup | are 0.01–0.02 differences real? | Efron & Tibshirani 1993; Dietterich 1998 |
| Power loss | **k accuracy**, mean absolute error of the estimated loss (% of P_rated), total-loss error | how classification mistakes become wrong watts | – (this project) |

**Self-check:** the macro-F1 recomputed here is compared with the one logged when the run was trained (`matches_log`).
A mismatch means the checkpoint file was replaced later (this happened to R05–R10 when Notebook 05 was run twice in October; their re-runs reused the same file name).

Outputs: `metrics_all.csv` (one row per run), `metrics_per_class.csv`, `metrics_summary.csv` (mean ± SD over seeds), `mcnemar_vs_best.csv`, `metrics_ensemble.csv`, `comparison_with_papers.csv`, and `test_probs/<RunID>.npy`.
**Run this after Notebook 06 has finished** (it needs its checkpoints). Takes a few minutes.

## 0. Setup and data (same split and preprocessing as Notebooks 05 and 06)

In [ ]:
import time, json
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.models import (resnet18, ResNet18_Weights, resnet50, ResNet50_Weights,
                                efficientnet_b0, EfficientNet_B0_Weights)
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, f1_score,
                             matthews_corrcoef, cohen_kappa_score, average_precision_score, roc_auc_score,
                             top_k_accuracy_score, log_loss)

REL = Path("2020-02-14_InfraredSolarModules/InfraredSolarModules")
CANDIDATES = [Path("../archive"), Path("../data/archive"), Path("archive"), Path("../../archive")]
DATA_DIR = next((r / REL for r in CANDIDATES if (r / REL / "images").exists()), None)
assert DATA_DIR is not None, "data folder not found"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

split = pd.read_csv("splits.csv")
GROUP = {"Cell": "Cell-level", "Cell-Multi": "Cell-level", "Cracking": "Cell-level", "Hot-Spot": "Cell-level",
         "Hot-Spot-Multi": "Cell-level", "Diode": "Substring-level", "Diode-Multi": "Substring-level",
         "Offline-Module": "Module-level", "Shadowing": "External", "Vegetation": "External", "Soiling": "External",
         "No-Anomaly": "No-Anomaly"}
CLASSES12 = sorted(split.label.unique())
CLASSES5 = ["Cell-level", "Substring-level", "Module-level", "External", "No-Anomaly"]
test_idx = np.where(split.split.values == "test")[0]
imgs = np.stack([np.asarray(Image.open(DATA_DIR / p), dtype=np.float32) / 255.0 for p in split.path.values[test_idx]])
med = np.median(imgs, axis=(1, 2), keepdims=True); std = imgs.std(axis=(1, 2), keepdims=True) + 1e-6
ZT = torch.from_numpy((imgs - med) / std).unsqueeze(1)             # (3000, 1, 40, 24) test images only
labels_test = split.label.values[test_idx]
print("test images:", tuple(ZT.shape))

## 1. Model definitions (copied from Notebooks 05 and 06, so every checkpoint can be rebuilt)

In [ ]:
# ---------- Notebook 05 models ----------
def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))
class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(conv_block(1, 32), conv_block(32, 64), conv_block(64, 128))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128 * 5 * 3, 256), nn.ReLU(),
                                        nn.Dropout(0.3), nn.Linear(256, n_classes))
    def forward(self, x):
        return self.classifier(self.features(x))
class ResNet18TL(nn.Module):
    def __init__(self, n_classes, size=(160, 96)):
        super().__init__()
        self.size = size; self.net = resnet18(weights=None)
        self.net.fc = nn.Linear(self.net.fc.in_features, n_classes)
    def forward(self, x):
        return self.net(F.interpolate(x, size=self.size, mode="bilinear", align_corners=False).repeat(1, 3, 1, 1))
class PretrainedTL(nn.Module):
    def __init__(self, arch, n_classes, size=(160, 96)):
        super().__init__()
        self.size = size
        if arch == "resnet50":
            self.net = resnet50(weights=None); self.net.fc = nn.Linear(self.net.fc.in_features, n_classes)
        else:
            self.net = efficientnet_b0(weights=None)
            self.net.classifier[1] = nn.Linear(self.net.classifier[1].in_features, n_classes)
    def forward(self, x):
        return self.net(F.interpolate(x, size=self.size, mode="bilinear", align_corners=False).repeat(1, 3, 1, 1))

# ---------- Notebook 06 models ----------
class ResNet18S(nn.Module):
    # ResNet-18 adapted to small images (He et al. 2016, CIFAR-10 setting): conv1 3x3 stride 1, no max-pool
    # 40x24 -> layer1 40x24 -> layer2 20x12 -> layer3 10x6 -> layer4 5x3 -> global avg-pool -> fc
    def __init__(self, n_classes, pretrained=True, dropout=0.0):
        super().__init__()
        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        w7 = net.conv1.weight.data.clone()                             # (64, 3, 7, 7) pretrained
        net.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        if pretrained:                                                 # centre 3x3 of each 7x7 filter, RGB summed
            net.conv1.weight.data.copy_(w7[:, :, 2:5, 2:5].sum(dim=1, keepdim=True))
        net.maxpool = nn.Identity()
        net.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(512, n_classes))
        self.net = net
    def forward(self, x):
        return self.net(x)

class WideBasic(nn.Module):
    # pre-activation block of Wide ResNet: BN-ReLU-conv3x3 -> (dropout) -> BN-ReLU-conv3x3, plus shortcut
    def __init__(self, c_in, c_out, stride, dropout):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(c_in), nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(c_out), nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.short = None if (stride == 1 and c_in == c_out) else nn.Conv2d(c_in, c_out, 1, stride, bias=False)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.short is None else self.short(o)
        o = self.conv1(o)
        o = self.conv2(self.drop(F.relu(self.bn2(o))))
        return o + sc

class WRN(nn.Module):
    # Wide ResNet (Zagoruyko & Komodakis 2016). WRN-16-4: 3 groups x 2 blocks, widths 64/128/256
    # 40x24 -> group1 40x24 -> group2 20x12 -> group3 10x6 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=16, k=4, dropout=0.0):
        super().__init__()
        n = (depth - 4) // 6
        widths = [16, 16 * k, 32 * k, 64 * k]
        self.conv = nn.Conv2d(1, widths[0], 3, 1, 1, bias=False)
        blocks, c_in = [], widths[0]
        for g, stride in enumerate([1, 2, 2]):
            for i in range(n):
                blocks.append(WideBasic(c_in, widths[g + 1], stride if i == 0 else 1, dropout))
                c_in = widths[g + 1]
        self.blocks = nn.Sequential(*blocks)
        self.bn = nn.BatchNorm2d(c_in)
        self.fc = nn.Linear(c_in, n_classes)
    def forward(self, x):
        o = F.relu(self.bn(self.blocks(self.conv(x))))
        return self.fc(F.adaptive_avg_pool2d(o, 1).flatten(1))

def conv_bn(c_in, c_out, k=3):
    return nn.Sequential(nn.Conv2d(c_in, c_out, k, padding=k // 2, bias=False), nn.BatchNorm2d(c_out), nn.ReLU())

class MSCNN(nn.Module):
    # multi-scale CNN, idea of Korkmaz & Acikgoz 2022 (three branches with different kernel sizes);
    # simplified re-implementation. branches 3x3 / 5x5 / 7x7 at 40x24 -> concat 96 ch
    # -> 128 ch, pool 20x12 -> 128 ch, pool 10x6 -> 256 ch -> global avg-pool -> fc
    def __init__(self, n_classes, dropout=0.0):
        super().__init__()
        self.branches = nn.ModuleList([nn.Sequential(conv_bn(1, 32, k), conv_bn(32, 32, k)) for k in (3, 5, 7)])
        self.body = nn.Sequential(conv_bn(96, 128), nn.MaxPool2d(2), conv_bn(128, 128), nn.MaxPool2d(2),
                                  conv_bn(128, 256))
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, n_classes))
    def forward(self, x):
        o = torch.cat([b(x) for b in self.branches], dim=1)
        o = self.body(o)
        return self.head(F.adaptive_avg_pool2d(o, 1).flatten(1))

class CifarBasic(nn.Module):
    # basic residual block of the CIFAR ResNets (He et al. 2016)
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.c1, self.b1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False), nn.BatchNorm2d(c_out)
        self.c2, self.b2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm2d(c_out)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out) else
                      nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out)))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x)))
        return F.relu(self.b2(self.c2(o)) + self.short(x))

class ResNetCifar(nn.Module):
    # ResNet-20 for small images (He et al. 2016, CIFAR-10): 3 stages x 3 blocks, widths 16/32/64
    # 40x24 -> stage1 40x24 -> stage2 20x12 -> stage3 10x6 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=20, dropout=0.0):
        super().__init__()
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(1, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        layers, c_in = [], 16
        for c_out, stride in [(16, 1), (32, 2), (64, 2)]:
            for i in range(n):
                layers.append(CifarBasic(c_in, c_out, stride if i == 0 else 1)); c_in = c_out
        self.layers = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        o = self.layers(self.stem(x))
        return self.head(F.adaptive_avg_pool2d(o, 1).flatten(1))

class DenseLayer(nn.Module):
    # DenseNet-BC bottleneck layer: BN-ReLU-conv1x1 (4k) -> BN-ReLU-conv3x3 (k), output concatenated to its input
    def __init__(self, c_in, k):
        super().__init__()
        self.f = nn.Sequential(nn.BatchNorm2d(c_in), nn.ReLU(), nn.Conv2d(c_in, 4 * k, 1, bias=False),
                               nn.BatchNorm2d(4 * k), nn.ReLU(), nn.Conv2d(4 * k, k, 3, 1, 1, bias=False))
    def forward(self, x):
        return torch.cat([x, self.f(x)], dim=1)

class DenseNetBC(nn.Module):
    # DenseNet-BC for small images (Huang et al. 2017), L = 100, k = 12, compression 0.5
    # 40x24 -> block1 40x24 -> transition 20x12 -> block2 -> transition 10x6 -> block3 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=100, k=12, dropout=0.0):
        super().__init__()
        n = (depth - 4) // 6
        c = 2 * k
        mods = [nn.Conv2d(1, c, 3, 1, 1, bias=False)]
        for b in range(3):
            for _ in range(n):
                mods.append(DenseLayer(c, k)); c += k
            if b < 2:
                c_out = c // 2
                mods += [nn.BatchNorm2d(c), nn.ReLU(), nn.Conv2d(c, c_out, 1, bias=False), nn.AvgPool2d(2)]
                c = c_out
        mods += [nn.BatchNorm2d(c), nn.ReLU()]
        self.features = nn.Sequential(*mods)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(c, n_classes))
    def forward(self, x):
        return self.head(F.adaptive_avg_pool2d(self.features(x), 1).flatten(1))

def build_model(name, n_classes, dropout):
    return {"ResNet18S": lambda: ResNet18S(n_classes, pretrained=True, dropout=dropout),
            "ResNet18S_scratch": lambda: ResNet18S(n_classes, pretrained=False, dropout=dropout),
            "WRN16_4": lambda: WRN(n_classes, 16, 4, dropout=dropout),
            "MSCNN": lambda: MSCNN(n_classes, dropout=dropout),
            "ResNet20": lambda: ResNetCifar(n_classes, 20, dropout=dropout),
            "DenseNetBC100": lambda: DenseNetBC(n_classes, 100, 12, dropout=dropout)}[name]()

MODEL_INFO = {
    "ResNet18S": "ResNet-18, small-image stem (conv1 3x3 s1, no max-pool; He et al. 2016 CIFAR setting), 1-channel 40x24 input, "
                 "layer1-4 ImageNet-pretrained (IMAGENET1K_V1), conv1 from centre 3x3 of pretrained 7x7 (RGB summed), all layers fine-tuned",
    "ResNet18S_scratch": "ResNet-18, small-image stem (conv1 3x3 s1, no max-pool), 1-channel 40x24 input, random initialisation",
    "WRN16_4": "Wide ResNet-16-4 (Zagoruyko & Komodakis 2016), pre-activation blocks, widths 64/128/256, 1-channel 40x24 input, random initialisation",
    "ResNet20": "ResNet-20 for small images (He et al. 2016, CIFAR-10): 3 stages x 3 basic blocks, widths 16/32/64, 1-channel 40x24 input, random initialisation",
    "DenseNetBC100": "DenseNet-BC L=100 k=12 for small images (Huang et al. 2017), compression 0.5, 1-channel 40x24 input, random initialisation",
    "MSCNN": "Multi-scale CNN (idea of Korkmaz & Acikgoz 2022, simplified): branches 3x3/5x5/7x7 (32 ch each) -> 128 -> pool -> 128 -> pool -> 256 -> GAP -> fc, random initialisation",
}
LR = {"ResNet18S": 3e-4, "ResNet18S_scratch": 1e-3, "WRN16_4": 1e-3, "MSCNN": 1e-3, "ResNet20": 1e-3, "DenseNetBC100": 1e-3}       # pretrained: smaller steps
EPOCHS = {"ResNet18S": 30, "ResNet18S_scratch": 40, "WRN16_4": 40, "MSCNN": 40, "ResNet20": 40, "DenseNetBC100": 40}           # from scratch: more epochs
RECIPES = {
    "base": dict(label_smoothing=0.0, dropout=0.0, weight_decay=1e-4, noise_std=0.0),
    "reg":  dict(label_smoothing=0.1, dropout=0.3, weight_decay=5e-4, noise_std=0.05),
}

def build_any(name, n_classes):
    nb05 = {"SmallCNN": lambda: SmallCNN(n_classes), "ResNet18TL": lambda: ResNet18TL(n_classes),
            "ResNet50TL": lambda: PretrainedTL("resnet50", n_classes), "EffNetB0TL": lambda: PretrainedTL("efficientnet_b0", n_classes)}
    if name in nb05:
        return nb05[name]()
    if name in ("ResNet18S", "ResNet18S_scratch"):
        return ResNet18S(n_classes, pretrained=False)              # weights come from the checkpoint
    return build_model(name, n_classes, 0.0)                         # dropout is inactive at evaluation

## 2. Metric functions
mAP here is the **classification** mAP: for each class, the average precision of its one-vs-rest precision–recall curve
(scikit-learn `average_precision_score`), then the mean over classes. ECE uses 15 equal-width confidence bins (Guo et al. 2017).

In [ ]:
def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(ece)

def all_metrics(probs, y, classes):
    K = len(classes); pred = probs.argmax(1); names = np.array(classes)
    onehot = np.eye(K)[y]
    P, R, F1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    m = {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
         "macro_precision": P, "macro_recall": R, "macro_F1": F1,
         "weighted_F1": f1_score(y, pred, average="weighted"),
         "MCC": matthews_corrcoef(y, pred), "cohen_kappa": cohen_kappa_score(y, pred),
         "mAP": average_precision_score(onehot, probs, average="macro"),
         "macro_ROC_AUC": roc_auc_score(y, probs, multi_class="ovr", average="macro"),
         "top2_accuracy": top_k_accuracy_score(y, probs, k=2, labels=range(K)),
         "test_NLL": log_loss(y, np.clip(probs, 1e-7, 1), labels=range(K)), "ECE": ece_score(probs, y)}
    true_fault, pred_fault = names[y] != "No-Anomaly", names[pred] != "No-Anomaly"
    fp, fr, ff, _ = precision_recall_fscore_support(true_fault, pred_fault, average="binary", zero_division=0)
    m.update(fault_accuracy=float(np.mean(true_fault == pred_fault)), fault_precision=fp, fault_recall=fr, fault_F1=ff)
    if K == 12:
        m["macro_F1_as_5_groups"] = f1_score([GROUP[c] for c in names[y]], [GROUP[c] for c in names[pred]], average="macro")
    else:
        m["macro_F1_as_5_groups"] = F1
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=range(K), zero_division=0)
    ap = [average_precision_score(onehot[:, k], probs[:, k]) for k in range(K)]
    per_class = pd.DataFrame({"class": classes, "precision": p, "recall": r, "F1": f, "AP": ap, "test_images": s})
    return m, per_class

def predict_probs(model, X, tta=False):
    # tta = test-time augmentation: average the softmax of the image and its 3 flipped copies
    outs = []
    with torch.no_grad():
        for i in range(0, len(X), 512):
            xb = X[i:i + 512].to(device)
            if tta:
                views = [xb, xb.flip(3), xb.flip(2), xb.flip(2).flip(3)]
                p = torch.stack([torch.softmax(model(v).float(), 1) for v in views]).mean(0)
            else:
                p = torch.softmax(model(xb).float(), 1)
            outs.append(p.cpu())
    return torch.cat(outs).numpy()

def paper_protocols(probs, y, classes):
    # the same numbers the papers report, computed from our predictions
    res = {}
    if len(classes) == 12:
        na = classes.index("No-Anomaly"); mask = y != na
        p11 = probs[mask].copy(); p11[:, na] = -1.0                 # model must choose among the 11 anomaly classes
        pred11 = p11.argmax(1)
        res["acc_11_anomaly_classes"] = accuracy_score(y[mask], pred11)
        res["macroF1_11_anomaly_classes"] = f1_score(y[mask], pred11, average="macro",
                                                     labels=[k for k in range(12) if k != na])
    return res

def fast_macro_f1(y, pred, K):
    cm = np.bincount(y * K + pred, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); prec = tp / np.maximum(cm.sum(0), 1); rec = tp / np.maximum(cm.sum(1), 1)
    return float(np.mean(np.where(prec + rec > 0, 2 * prec * rec / np.maximum(prec + rec, 1e-12), 0)))

def bootstrap_ci(y, pred, K, n=1000, seed=0):
    # 95 % confidence interval of macro-F1: resample the 3,000 test images with replacement (Efron & Tibshirani 1993)
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y)); vals.append(fast_macro_f1(y[i], pred[i], K))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

# number of bypassed substrings k implied by each class (power loss = k/3 of P_rated).
# Classes whose loss is not estimated this term (cell-level, external) count as k = 0 here, so their error shows as under-estimation.
K_OF = {"No-Anomaly": 0, "Diode": 1, "Diode-Multi": 2, "Offline-Module": 3}
def power_loss_metrics(y, pred, classes):
    names = np.array(classes)
    kt = np.array([K_OF.get(c, 0) for c in names[y]]); kp = np.array([K_OF.get(c, 0) for c in names[pred]])
    lt, lp = kt / 3 * 100, kp / 3 * 100                                  # loss in % of P_rated per module
    sub = np.isin(names[y], ["Diode", "Diode-Multi", "Offline-Module"])     # modules with a substring/module-level loss
    return {"k_accuracy": float(np.mean(kt == kp)),
            "loss_MAE_pct_all": float(np.mean(np.abs(lp - lt))),
            "loss_MAE_pct_on_lossy_modules": float(np.mean(np.abs(lp[sub] - lt[sub]))),
            "total_loss_error_pct": float((lp.sum() - lt.sum()) / max(lt.sum(), 1e-9) * 100)}

## 3. Find every run and its checkpoint
Runs come from `results_05.csv` and `results_06.csv`. The checkpoint path and settings are read from each run's `config.json`.

In [ ]:
rows = []
for f, nb in (("results_05.csv", "05"), ("results_06.csv", "06")):
    if not Path(f).exists():
        print(f, "not found, skipped"); continue
    for _, r in pd.read_csv(f).iterrows():
        cfgs = sorted(Path("runs").glob(f"{r.run_id}_*/config.json"))
        cfg = json.loads(cfgs[-1].read_text(encoding="utf-8")) if cfgs else {}
        ckpt = Path(cfg.get("checkpoint", "")) if cfg.get("checkpoint") else None
        if ckpt is None or not ckpt.exists():                       # fall back to the naming pattern of the notebook
            hits = sorted(Path("checkpoints").glob(f"{r.run_id}_*.pt"))
            ckpt = hits[0] if hits else None
        classes = str(r.get("classes", "12")) if nb == "05" else "12"
        setup = (f"{r.model} | {classes} cls | {r.balance}" if nb == "05" else f"{r.model} | {r.recipe} (preserve 40x24)")
        rows.append(dict(notebook=nb, run_id=r.run_id, rerun_of=r.get("rerun_of"), model=r.model, classes=classes,
                         setup=setup, seed=int(r.seed) if not pd.isna(r.get("seed", np.nan)) else 42,
                         logged_macro_F1=r.test_f1, checkpoint=str(ckpt) if ckpt else None))
runs = pd.DataFrame(rows)
print(len(runs), "runs,", runs.checkpoint.notna().sum(), "with a checkpoint")
print(runs[runs.checkpoint.isna()].run_id.tolist() or "all checkpoints found")

## 4. Evaluate every checkpoint (test set only, no training)

In [ ]:
Path("test_probs").mkdir(exist_ok=True)
out, per_class_all = [], []
for _, r in runs[runs.checkpoint.notna()].iterrows():
    classes = CLASSES12 if r.classes == "12" else CLASSES5
    y = np.array([classes.index(c if r.classes == "12" else GROUP[c]) for c in labels_test])
    model = build_any(r.model, len(classes)).to(device)
    try:
        model.load_state_dict(torch.load(r.checkpoint, map_location=device))
    except Exception as e:
        print(f"{r.run_id}: cannot load {r.checkpoint} ({type(e).__name__})"); continue
    model.eval()
    with torch.no_grad():
        model(ZT[:64].to(device))                                    # warm-up
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    probs = predict_probs(model, ZT)
    if device.type == "cuda": torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000 / len(ZT)
    probs_tta = predict_probs(model, ZT, tta=True)
    np.save(Path("test_probs") / f"{r.run_id}_tta.npy", probs_tta)
    np.save(Path("test_probs") / f"{r.run_id}.npy", probs)
    m, pc = all_metrics(probs, y, classes)
    pred = probs.argmax(1)
    m["macro_F1_CI_low"], m["macro_F1_CI_high"] = bootstrap_ci(y, pred, len(classes))
    m.update(paper_protocols(probs, y, classes))
    if r.classes == "12":
        m.update(power_loss_metrics(y, pred, classes))
    mt, _ = all_metrics(probs_tta, y, classes); mt.update(paper_protocols(probs_tta, y, classes))
    for k in ("accuracy", "macro_F1", "macro_F1_as_5_groups", "mAP", "MCC", "test_NLL", "ECE", "fault_accuracy", "acc_11_anomaly_classes"):
        if k in mt: m["tta_" + k] = mt[k]
    np.save(Path("test_probs") / f"{r.run_id}_true.npy", y)
    m.update(run_id=r.run_id, notebook=r.notebook, setup=r.setup, model=r.model, classes=r.classes, seed=r.seed,
             rerun_of=r.rerun_of, params=sum(p.numel() for p in model.parameters()), infer_ms_per_image=ms,
             logged_macro_F1=r.logged_macro_F1, matches_log=bool(abs(m["macro_F1"] - r.logged_macro_F1) < 0.005))
    out.append(m); pc.insert(0, "run_id", r.run_id); per_class_all.append(pc)
    print(f"{r.run_id:12s} {r.setup:45s} acc {m['accuracy']:.3f}  macro-F1 {m['macro_F1']:.3f}  mAP {m['mAP']:.3f}  "
          f"MCC {m['MCC']:.3f}  ECE {m['ECE']:.3f}  {'' if m['matches_log'] else '<-- differs from log'}")

first = ["run_id", "notebook", "setup", "seed", "matches_log", "logged_macro_F1"]
metrics = pd.DataFrame(out); metrics = metrics[first + [c for c in metrics.columns if c not in first]]
metrics.round(4).to_csv("metrics_all.csv", index=False)
pd.concat(per_class_all).round(4).to_csv("metrics_per_class.csv", index=False)
print("\nsaved metrics_all.csv and metrics_per_class.csv")

## 5. Mean ± SD over seeds for each setup (originals only, checkpoints that match their log)

In [ ]:
keep = metrics[metrics.rerun_of.isna() & metrics.matches_log]
cols = ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_F1", "weighted_F1", "MCC", "mAP",
        "macro_ROC_AUC", "test_NLL", "ECE", "fault_recall", "fault_F1", "macro_F1_as_5_groups", "infer_ms_per_image",
        "k_accuracy", "loss_MAE_pct_all", "loss_MAE_pct_on_lossy_modules", "total_loss_error_pct",
        "acc_11_anomaly_classes", "macroF1_11_anomaly_classes",
        "tta_accuracy", "tta_macro_F1", "tta_macro_F1_as_5_groups", "tta_test_NLL", "tta_acc_11_anomaly_classes"]
g = keep.groupby("setup")
summary = g[[c for c in cols if c in keep]].mean().round(3)
for c in ("macro_F1", "mAP", "MCC", "macro_F1_as_5_groups", "test_NLL"):
    summary[c + "_sd"] = g[c].std().round(3)
summary.insert(0, "runs", g.run_id.apply(", ".join)); summary.insert(1, "n", g.size()); summary.insert(2, "params", g.params.first())
summary = summary.sort_values("macro_F1_as_5_groups", ascending=False)
summary.to_csv("metrics_summary.csv")
pd.set_option("display.width", 250)
print(summary[["runs", "n", "params", "accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_F1",
               "macro_F1_sd", "mAP", "MCC", "test_NLL", "ECE", "fault_recall", "macro_F1_as_5_groups"]].to_string())
excluded = metrics[~metrics.matches_log].run_id.tolist()
print("\nexcluded (checkpoint replaced after logging):", excluded or "none")

## 6. Is the difference real? McNemar test (seed-42 run of every setup vs the best setup)
McNemar's test (Dietterich 1998) compares two classifiers **on the same test images**: it only looks at the images where
one model is right and the other is wrong (b and c). p < 0.05 → the two models really differ; otherwise the difference may be chance.
The bootstrap 95 % interval of each run's macro-F1 is in `metrics_all.csv` (`macro_F1_CI_low`, `macro_F1_CI_high`).

In [ ]:
from scipy.stats import binomtest
s42 = keep[(keep.seed == 42) & (keep.classes == "12")].set_index("setup")
best_setup = summary[summary.index.isin(s42.index)].index[0]
ref = s42.loc[best_setup, "run_id"]
y_true = np.load(Path("test_probs") / f"{ref}_true.npy")
p_ref = np.load(Path("test_probs") / f"{ref}.npy").argmax(1)
rows = []
for setup, row in s42.iterrows():
    if setup == best_setup: continue
    p = np.load(Path("test_probs") / f"{row.run_id}.npy").argmax(1)
    b = int(np.sum((p_ref == y_true) & (p != y_true)))      # best right, other wrong
    c = int(np.sum((p_ref != y_true) & (p == y_true)))      # best wrong, other right
    pval = binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0
    rows.append(dict(setup=setup, run=row.run_id, vs=f"{best_setup} ({ref})", best_right_other_wrong=b,
                     best_wrong_other_right=c, p_value=round(pval, 4), significant=pval < 0.05))
mcn = pd.DataFrame(rows).sort_values("p_value")
mcn.to_csv("mcnemar_vs_best.csv", index=False)
print("best setup:", best_setup); print(mcn.to_string(index=False))

## 7. Seed ensemble: average the probabilities of the 3 seeds of each setup
Le et al. (2021) used an ensemble of residual networks on this dataset. The cheapest ensemble is already available:
the 3 seed runs of every setup are 3 different models. Their softmax outputs are averaged (with and without test-time augmentation).
No training is needed. Cost: 3 × the inference time.

In [ ]:
ens_rows = []
for setup, g_ in keep.groupby("setup"):
    if len(g_) < 2: continue
    classes = CLASSES12 if g_.classes.iloc[0] == "12" else CLASSES5
    y = np.load(Path("test_probs") / f"{g_.run_id.iloc[0]}_true.npy")
    for tta in (False, True):
        P = np.mean([np.load(Path("test_probs") / f"{rid}{'_tta' if tta else ''}.npy") for rid in g_.run_id], axis=0)
        m, _ = all_metrics(P, y, classes); m.update(paper_protocols(P, y, classes))
        if len(classes) == 12: m.update(power_loss_metrics(y, P.argmax(1), classes))
        m["macro_F1_CI_low"], m["macro_F1_CI_high"] = bootstrap_ci(y, P.argmax(1), len(classes))
        m.update(setup=setup, ensemble=f"{len(g_)} seeds" + (" + TTA" if tta else ""), runs=", ".join(g_.run_id),
                 single_model_macro_F1_mean=g_.macro_F1.mean())
        ens_rows.append(m)
ens = pd.DataFrame(ens_rows)
ens = ens[["setup", "ensemble", "runs", "single_model_macro_F1_mean"] + [c for c in ens.columns if c not in ("setup", "ensemble", "runs", "single_model_macro_F1_mean")]]
ens = ens.sort_values("macro_F1_as_5_groups", ascending=False)
ens.round(4).to_csv("metrics_ensemble.csv", index=False)
print(ens[["setup", "ensemble", "single_model_macro_F1_mean", "accuracy", "macro_F1", "macro_F1_CI_low", "macro_F1_CI_high",
           "macro_F1_as_5_groups", "mAP", "test_NLL", "ECE"]].round(3).head(15).to_string(index=False))

## 8. Comparison with published results on the same dataset, computed the way each paper reports
The papers do not all measure the same thing. To compare fairly, our predictions are scored with **each paper's protocol**:
* **12-class accuracy** (Le et al. 2021),
* **11 anomaly classes**: only the faulty test images, and the model must choose among the 11 anomaly classes (Korkmaz & Açıkgöz 2022; Fonseca Alves et al. 2021 report 8 selected classes, so their number is not directly comparable),
* **fault / no-fault accuracy** (all three papers).

Paper values are copied from the papers as reported (accuracy). Differences that remain are partly due to setup:
ensembles, how the test set was built (natural class mix here, never resampled), and whether augmented copies were made before or after splitting.

In [ ]:
papers = pd.DataFrame([
    dict(source="Fonseca Alves et al. 2021 (Renewable Energy)", model="CNN", acc_12=np.nan, acc_11_anomaly=np.nan, acc_fault=0.925, macro_F1_12=np.nan,
         notes="as reported in the abstract: 92.5 % fault/no-fault (cross-validation); 78.85 % on 8 selected anomaly classes"),
    dict(source="Le et al. 2021 (SETA)", model="ResNet-style ensemble", acc_12=0.86, acc_11_anomaly=np.nan, acc_fault=0.94, macro_F1_12=np.nan,
         notes="as reported (≈)"),
    dict(source="Korkmaz & Açıkgöz 2022 (EAAI)", model="multi-scale CNN + transfer learning", acc_12=np.nan, acc_11_anomaly=0.935, acc_fault=0.973,
         macro_F1_12=np.nan, notes="as reported; balanced by offline augmentation"),
])
def our_row(label, d, n_note):
    return dict(source=label, model=n_note, acc_12=d.get("accuracy"), acc_11_anomaly=d.get("acc_11_anomaly_classes"),
                acc_fault=d.get("fault_accuracy"), macro_F1_12=d.get("macro_F1"), notes="this work, test set never resampled")
best_single = summary[summary.n >= 1].index[0]
s_best = keep[keep.setup == best_single][["accuracy", "acc_11_anomaly_classes", "fault_accuracy", "macro_F1"]].mean().to_dict()
rows_ = [our_row(f"This work: {best_single} (single model, mean of {int(summary.loc[best_single, 'n'])} seeds)", s_best, "single model")]
e12 = ens[ens.setup.isin(keep[keep.classes == "12"].setup)]
if len(e12):
    b = e12.iloc[0]
    rows_.append(our_row(f"This work: {b.setup} ({b.ensemble})", b.to_dict(), "seed ensemble"))
comparison = pd.concat([papers, pd.DataFrame(rows_)], ignore_index=True)
comparison.round(3).to_csv("comparison_with_papers.csv", index=False)
print(comparison.round(3).to_string(index=False))

## References for the metrics
- Sokolova, M., & Lapalme, G. (2009). A systematic analysis of performance measures for classification tasks. *Information Processing & Management*, 45(4), 427–437.
- Grandini, M., Bagli, E., & Visani, G. (2020). Metrics for multi-class classification: an overview. arXiv:2008.05756.
- Brodersen, K. H., Ong, C. S., Stephan, K. E., & Buhmann, J. M. (2010). The balanced accuracy and its posterior distribution. *ICPR 2010*, 3121–3124.
- Matthews, B. W. (1975). Comparison of the predicted and observed secondary structure of T4 phage lysozyme. *Biochimica et Biophysica Acta*, 405(2), 442–451.
- Chicco, D., & Jurman, G. (2020). The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation. *BMC Genomics*, 21, 6.
- Cohen, J. (1960). A coefficient of agreement for nominal scales. *Educational and Psychological Measurement*, 20(1), 37–46.
- Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLoS ONE*, 10(3), e0118432.
- Everingham, M., Van Gool, L., Williams, C. K. I., Winn, J., & Zisserman, A. (2010). The PASCAL Visual Object Classes (VOC) Challenge. *IJCV*, 88(2), 303–338.
- Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). On calibration of modern neural networks. *ICML 2017*.
- Dietterich, T. G. (1998). Approximate statistical tests for comparing supervised classification learning algorithms. *Neural Computation*, 10(7), 1895–1923.
- Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.
- Le, M., et al. (2021). Remote anomaly detection and classification of solar photovoltaic modules based on deep neural network. *Sustainable Energy Technologies and Assessments*, 48, 101545.
- Korkmaz, D., & Açıkgöz, H. (2022). An efficient fault classification method in solar photovoltaic modules using transfer learning and multi-scale convolutional neural network. *Engineering Applications of Artificial Intelligence*, 113, 104959.
- Fonseca Alves, R. H., de Deus Júnior, G. A., Marra, E. G., & Lemos, R. P. (2021). Automatic fault classification in photovoltaic modules using Convolutional Neural Networks. *Renewable Energy*, 179, 502–516.